# 03 — Backtest: ridge vs the transparent baseline

Rolling-origin evaluation on a single grid cell. Reports MAE / RMSE / **PicP** (p10–p90 coverage)
for the trained ridge forecaster and for the `linear_trend_seasonal` baseline.

If the ridge does not beat the baseline on your data, believe the baseline and investigate.
The bundled store is synthetic, so these numbers validate the *plumbing*, not climate skill.

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
from pipeline.export_store import load_store
from pipeline.build_features import prepare_cell_matrix
from models.backtest import backtest_cell, mae, rmse, picp
from models.forecast import linear_trend_seasonal_forecast

store = load_store('../data/samples/earth_pulse_sample.npz')
variables = ['temperature', 'precipitation', 'ndvi']
cell = 134
dates, X = prepare_cell_matrix(store, cell, variables)
print('cell', cell, 'lat', store['grid_lat'][cell], 'lon', store['grid_lon'][cell])

horizon = 24
seq_len = 24
train_end = len(dates) - horizon
print(f'train: {dates[0]} → {dates[train_end - 1]} ({train_end} months)')
print(f'holdout: {dates[train_end]} → {dates[-1]} ({horizon} months)')

In [ ]:
# 1) Ridge rollout, trained only on the prefix
ridge = backtest_cell(X, variables, seq_len=seq_len, test_months=horizon, horizon=horizon)
print('ridge_rollout — rolling-origin holdout')
for name, m in ridge['per_parameter'].items():
    print(f"  {name:14s} MAE={m['mae']:.4f}  RMSE={m['rmse']:.4f}  PicP(p10-p90)={m['picp_p10_p90']:.2f}  bias={m['bias']:+.4f}")

# 2) Transparent baseline: Theil–Sen trend + monthly climatology
print('\nlinear_trend_seasonal — same holdout')
baseline = {}
for j, name in enumerate(variables):
    res = linear_trend_seasonal_forecast(X[:train_end, j], dates[:train_end], horizon)
    actual = X[train_end:train_end + horizon, j]
    mean, lo, hi = (np.asarray(res[k]) for k in ('mean', 'lower', 'upper'))
    baseline[name] = {
        'mae': mae(mean, actual),
        'rmse': rmse(mean, actual),
        'picp_p10_p90': picp(lo, hi, actual),
    }
    b = baseline[name]
    print(f"  {name:14s} MAE={b['mae']:.4f}  RMSE={b['rmse']:.4f}  PicP(p10-p90)={b['picp_p10_p90']:.2f}")

print('\nRMSE winner per parameter:')
for name in variables:
    r, b = ridge['per_parameter'][name]['rmse'], baseline[name]['rmse']
    print(f"  {name:14s} {'ridge' if r <= b else 'baseline'} (ridge={r:.4f}, baseline={b:.4f})")

### Reading PicP

PicP is the empirical coverage of the nominal 80% interval (p10–p90).
- **≈0.80** — bands are calibrated.
- **< 0.80** — bands are too narrow (overconfident); the API surfaces this so it is not hidden.
- **> 0.95** — bands are too wide (underconfident, low information).

Longer horizons require wider bands: the code inflates spread with `√h`, so PicP should stay
roughly stable as `horizon` grows if the residual model is reasonable.

In [ ]:
# Multi-origin sanity check: does the ridge stay ahead of the baseline across cut-offs?
origins = [12, 24, 36]
rows = []
for h in origins:
    r = backtest_cell(X, variables, seq_len=seq_len, test_months=h, horizon=h)
    acc = []
    for j, name in enumerate(variables):
        res = linear_trend_seasonal_forecast(X[:len(dates) - h, j], dates[:len(dates) - h], h)
        actual = X[len(dates) - h:, j]
        acc.append(rmse(np.asarray(res['mean']), actual) - r['per_parameter'][name]['rmse'])
    rows.append((h, float(np.mean(acc))))

print('horizon | mean(baseline RMSE - ridge RMSE)  [>0 means ridge wins]')
for h, delta in rows:
    print(f'  {h:>3d}   | {delta:+.4f}')